# Reproduce Ω figures (PJM)

Upload `rt_hrl_lmps.csv` when prompted, then run the notebook to reproduce Figure 1 and Figure 2.


In [ ]:
# Reproduce Ω figures (PJM)
# Upload rt_hrl_lmps.csv when prompted

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

rt = pd.read_csv(filename)

rt = rt[["datetime_beginning_ept", "pnode_name", "total_lmp_rt"]].copy()
rt.columns = ["datetime", "pnode", "rt_lmp"]

rt["datetime"] = pd.to_datetime(
    rt["datetime"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

rt = rt.dropna().sort_values(["datetime", "pnode"]).reset_index(drop=True)

rto = rt[rt["pnode"] == "PJM-RTO"][["datetime", "rt_lmp"]].copy()
rto.columns = ["datetime", "p_rto"]

zones = rt[rt["pnode"] != "PJM-RTO"].copy()

zone_stats = (
    zones.groupby("datetime")["rt_lmp"]
    .agg(
        sigma_zone="std",
        p50=lambda x: x.quantile(0.50),
        p95=lambda x: x.quantile(0.95),
        count="count"
    )
    .reset_index()
)

zone_stats = zone_stats[zone_stats["count"] > 10].copy()

df = zone_stats.merge(rto, on="datetime", how="inner").sort_values("datetime").reset_index(drop=True)

df["spread_95_rto"] = (df["p95"] - df["p_rto"]).clip(lower=0)
df["Omega_95"] = df["sigma_zone"] * df["spread_95_rto"]

df["ratio_95"] = df["p95"] / (df["p_rto"] + 1e-9)

thr_event = df["ratio_95"].quantile(0.998)
df["event"] = (df["ratio_95"] >= thr_event).astype(int)

df["event_lead6"] = df["event"].shift(-6)
df = df.dropna(subset=["event_lead6"]).copy()
df["event_lead6"] = df["event_lead6"].astype(int)

print("threshold for event =", thr_event)
print("event count =", int(df["event"].sum()))

tmp = df[["Omega_95", "event_lead6"]].copy()
tmp["omega_decile"] = pd.qcut(tmp["Omega_95"], 10, labels=False, duplicates="drop")

dec = (
    tmp.groupby("omega_decile")["event_lead6"]
    .agg(["mean", "sum", "count"])
    .reset_index()
    .rename(columns={"mean": "event_rate", "sum": "event_hits", "count": "n"})
)

print("\nFigure 1 table:")
print(dec)

plt.figure(figsize=(8, 5))
plt.plot(dec["omega_decile"], dec["event_rate"], marker="o")
plt.xlabel("Omega decile")
plt.ylabel("Event rate")
plt.title("Event rate by Omega decile")
plt.tight_layout()
plt.show()

top_bin = tmp["omega_decile"].max()
tmp["group"] = np.where(
    tmp["omega_decile"] == top_bin,
    "Top Omega decile",
    "Lower Omega deciles"
)

summary = (
    tmp.groupby("group")["event_lead6"]
    .agg(["mean", "sum", "count"])
    .reset_index()
    .rename(columns={"mean": "event_rate", "sum": "event_hits", "count": "n"})
)

print("\nFigure 2 table:")
print(summary)

plt.figure(figsize=(6, 5))
plt.bar(summary["group"], summary["event_rate"])

for i, row in summary.iterrows():
    plt.text(
        i,
        row["event_rate"] + 0.0003,
        f"rate={row['event_rate']:.4f}\nhits={int(row['event_hits'])}/{int(row['n'])}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.ylabel("Event rate")
plt.title("Event concentration in the highest Omega regime")
plt.tight_layout()
plt.show()
